# 01 - Carga dos dados
Fonte: [Sinistros de trânsito - Dados Abertos Fortaleza](https://dados.fortaleza.ce.gov.br/dataset/sinistros-transito) (AMC).

Aqui só carregamos a base bruta e olhamos como ela é. Nada é alterado.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # pra achar a pasta src

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

## Onde colocar o arquivo
Baixe os arquivos do site e coloque em `data/raw/` (essa pasta não vai pro GitHub porque a base é grande).
Se a base vier em vários arquivos (um por ano, por exemplo), o código abaixo junta todos.

In [ ]:
arquivos = sorted(list(RAW.glob("*.geojson")) + list(RAW.glob("*.shp")) + list(RAW.glob("*.gpkg")) + list(RAW.glob("*.csv")))
print(f"{len(arquivos)} arquivo(s) em data/raw:")
for a in arquivos:
    print(" -", a.name)

In [ ]:
partes = []
for a in arquivos:
    if a.suffix == ".csv":
        df_ = pd.read_csv(a, sep=None, engine="python")
        df_ = gpd.GeoDataFrame(df_, geometry=gpd.points_from_xy(df_["LONGITUDE"], df_["LATITUDE"]), crs="EPSG:4326")
    else:
        df_ = gpd.read_file(a)
    partes.append(df_)

df = pd.concat(partes, ignore_index=True)
df = gpd.GeoDataFrame(df, geometry="geometry", crs=partes[0].crs)
df.shape

## Primeira olhada

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.describe()

## Valores nulos

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

## Salvando a base bruta
Salvamos em parquet porque lê muito mais rápido que csv/geojson. Os próximos notebooks partem daqui.

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_parquet(ARQ_BRUTO)
print("salvo em", ARQ_BRUTO)